# 1. The data

**Question:** who are the patients in our dataset?

We use two hospitals from the UCI Heart Disease dataset (Cleveland and Hungary).
Together they give more patients, which we need to check the model for different groups.

## Read the two files
The files have no column names, so we give them. In these files `?` means *missing*.

In [ ]:
import pandas as pd
import numpy as np

columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]

cleveland = pd.read_csv("../data/processed.cleveland.data", header=None, names=columns, na_values="?")
hungary = pd.read_csv("../data/processed.hungarian.data", header=None, names=columns, na_values="?")

print("Cleveland:", cleveland.shape)
print("Hungary:  ", hungary.shape)

## Put the two hospitals together
We add a column `hospital` to remember where each patient comes from, then stack the two tables.

In [ ]:
cleveland["hospital"] = "Cleveland"
hungary["hospital"] = "Hungary"

data = pd.concat([cleveland, hungary], ignore_index=True)

print("All patients:", data.shape)

## Choose the columns and clean
- We keep 10 features. `slope`, `ca` and `thal` are almost always empty in Hungary, so we leave them out.
- `num` goes from 0 to 4 (disease severity). We turn it into **0 = healthy, 1 = sick**.
- We remove patients with any missing value.

In [ ]:
features = ["age", "sex", "cp", "trestbps", "chol", "fbs",
            "restecg", "thalach", "exang", "oldpeak"]

data["target"] = (data["num"] > 0).astype(int)

data = data[features + ["target", "hospital"]].dropna()

print("Patients after cleaning:", len(data))

## The groups we will check for fairness
- **Sex:** woman / man
- **Age:** 55 or younger / over 55 (we use 55 because Hungary has very few patients over 60)

For each group we look at its size and its share of sick patients.

In [ ]:
data["group_sex"] = data["sex"].map({0: "Woman", 1: "Man"})
data["group_age"] = np.where(data["age"] > 55, "Over 55", "55 or younger")

print("By sex:")
print(data["group_sex"].value_counts())
print()
print("By age:")
print(data["group_age"].value_counts())
print()
print("Share of sick patients in each group:")
print(data.groupby("group_sex")["target"].mean().round(3))
print(data.groupby("group_age")["target"].mean().round(3))

## Save the clean data
The next notebooks start from this file.

In [ ]:
data.to_csv("../data/clean.csv", index=False)
print("Saved", len(data), "patients")